# Export Pipeline - Production Build

Ce notebook a pour but de créer les artefacts 
 (Data, Scaler, Modèle ML) destinés à être embarqués dans l'API de production. 
Contrairement au notebook de modélisation (EDA), ici, nous n'appliquons que les opérations strictement nécessaires à l'inférence. L'objectif est d'obtenir le modèle le plus léger et le plus robuste possible.

In [ ]:
import pandas as pd
import joblib
import os
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

ROOT_DIR = Path('..').resolve()
DATA_DIR = ROOT_DIR / 'data'
MODEL_DIR = ROOT_DIR / 'models'

os.makedirs(MODEL_DIR, exist_ok=True)

### 1. Chargement et Nettoyage

In [2]:
# Chargement du jeu de données original
raw_data_path = DATA_DIR / 'dataset_2025_togo.csv'
df = pd.read_csv(raw_data_path)

print(f"Dimension originale : {df.shape}")

Dimension originale : (21000, 37)


In [3]:
# Colonnes à exclure impérativement du modèle d'entraînement selon notre EDA
cols_to_drop = [
    'system:index', '.geo', 'longitude', 'latitude', # Métadonnées et spatial pur
    'NDWI_mean', 'NDVI_mean', 'B4_mean',             # Trop corrélées aux indices de base
    'NDWI_var', 'NDVI_var',                          # Trop corrélées aux Contrastes
    'B12_ent', 'NDWI_ent', 'B8_ent', 'NDVI_ent', 'NDBI_ent', # Inverses de l'ASM
    'NDBI_asm', 'NDWI_asm', 'NDVI_asm', 'B12_asm'    # Très faible importance (< 0.02)
]

df_clean = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')

print(f"Dimension après nettoyage : {df_clean.shape}")
print(f"Colonnes conservées pour l'entraînement : \n{df_clean.columns.tolist()}")

Dimension après nettoyage : (21000, 19)
Colonnes conservées pour l'entraînement : 
['B12', 'B12_contrast', 'B12_diss', 'B4', 'B4_var', 'B8', 'B8_asm', 'B8_contrast', 'B8_diss', 'NDBI', 'NDBI_contrast', 'NDBI_diss', 'NDVI', 'NDVI_contrast', 'NDVI_diss', 'NDWI', 'NDWI_contrast', 'NDWI_diss', 'landcover']


### 2. Rééquilibrage métier (6 Classes)

In [4]:
# Fusion de la classe 'Savanes' (2) avec 'Buissons' (1)
df_clean['landcover'] = df_clean['landcover'].replace(2, 1)

# Recodage pour avoir une suite continue (0 à 5)
mapping = {0: 0, 1: 1, 3: 2, 4: 3, 5: 4, 6: 5}
df_clean['landcover'] = df_clean['landcover'].map(mapping)

# Noms officiels pour l'API
class_names = {
    0: 'Forêt', 
    1: 'Savanes/Buissons', 
    2: 'Cultures', 
    3: 'Urbain', 
    4: 'Sols nus', 
    5: 'Eau'
}

print("Classes prêtes pour la production :", class_names)

# --- SAUVEGARDE DU DATASET DE PRODUCTION ---
prod_dataset_path = DATA_DIR / 'dataset_clean_for_production.csv'
df_clean.to_csv(prod_dataset_path, index=False)
print(f"\n💾 Dataset allégé sauvegardé : {prod_dataset_path}")

Classes prêtes pour la production : {0: 'Forêt', 1: 'Savanes/Buissons', 2: 'Cultures', 3: 'Urbain', 4: 'Sols nus', 5: 'Eau'}

💾 Dataset allégé sauvegardé : /run/media/kjd/working/projects/IA/Deforestation/land-cover-ml-sentinel2/data/dataset_clean_for_production.csv


### 3. Entraînement Final (Scaler & Modèle)

In [6]:
X = df_clean.drop(columns=['landcover'])
y = df_clean['landcover']

# Entraînement du nouveau Scaler orienté production (Toutes les données)
scaler_prod = StandardScaler()
X_scaled = scaler_prod.fit_transform(X)

# Paramètres issus du GridSearchCV optimisé f1-macro
selected_model = joblib.load(MODEL_DIR / 'rfc_optimized-2_6classes.joblib')
final_params = selected_model.get_params()
rfc_prod = RandomForestClassifier(**final_params)

print("Entraînement du modèle de production sur l'intégralité du dataset...")
rfc_prod.fit(X_scaled, y)
print("✅ Entraînement terminé.")

y_preds = rfc_prod.predict(X_scaled)
print(f"Score sur train (Sanity Check) : {accuracy_score(y, y_preds):.3f}")

Entraînement du modèle de production sur l'intégralité du dataset...
✅ Entraînement terminé.
Score sur train (Sanity Check) : 0.977


### 4. Export des Artefacts

In [7]:
scaler_path = MODEL_DIR / 'scaler_production.joblib'
model_path = MODEL_DIR / 'rfc_production_v1.joblib'

joblib.dump(scaler_prod, scaler_path)
joblib.dump(rfc_prod, model_path)

print(f"💾 Scaler enregistré : {scaler_path}")
print(f"💾 Modèle enregistré : {model_path}")

💾 Scaler enregistré : /run/media/kjd/working/projects/IA/Deforestation/land-cover-ml-sentinel2/models/scaler_production.joblib
💾 Modèle enregistré : /run/media/kjd/working/projects/IA/Deforestation/land-cover-ml-sentinel2/models/rfc_production_v1.joblib
